# 02 · Uncertainty, calibration and token-pruning analysis

Reproduces Figures 3–7 of the paper with the trained Dynamic ViT
(main experiment, seed 5):

| Section | Figure |
|---|---|
| Predictive entropy vs per-class accuracy | Fig. 3 |
| Calibration (reliability diagram, class-wise ECE) | Fig. 4 |
| Sparsification process for the 20 fabrics | Fig. 5 |
| Token keep probability heatmaps | Fig. 6 |
| Token exit depth and token influence (Δprob) | Fig. 7 |

The trained checkpoint is downloaded automatically. The dataset is available on
request (see README). A GPU runtime is recommended.

## 1. Setup

In [ ]:
# Dependencies (timm 0.6.13 is required by the DynamicViT code)
!pip install -q tensorboardX gdown timm==0.6.13

# Official DynamicViT implementation (Rao et al., NeurIPS 2021, MIT License)
!git clone -q https://github.com/raoyongming/DynamicViT.git

# Compatibility fix: torch._six was removed in recent PyTorch versions
!sed -i 's/from torch._six import inf/inf = float("inf")/' DynamicViT/utils.py

import sys
sys.path.append("DynamicViT")
sys.path.append("DynamicViT/models")

In [ ]:
# Only on Colab, if the data are on Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

SEED = 5   # the figures in the paper refer to seed 5

DATA_DIR = "/content/drive/MyDrive/data"     # <-- change to your data folder
IMG_DIR = os.path.join(DATA_DIR, "DATASET")
CSV_FILE = os.path.join(DATA_DIR, "dataset.csv")

# Trained Dynamic ViT (seed 5), shared on Google Drive
CHECKPOINT_GDRIVE_ID = "https://drive.google.com/file/d/1UnUDhQyZtEgaVZZQyvon5F4s-CDpHYTQ/view"
CHECKPOINT_PATH = "checkpoints/DYNAMIC_VIT_model.pth"

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
import gdown

if not os.path.exists(CHECKPOINT_PATH):
    os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
    gdown.download(id=CHECKPOINT_GDRIVE_ID, output=CHECKPOINT_PATH, quiet=False)
print("Checkpoint:", CHECKPOINT_PATH, os.path.exists(CHECKPOINT_PATH))

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD

from dylvvit import LVViTDiffPruning, LVViT_Teacher

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 2. Data and model

In [ ]:
class FabricsDataset(Dataset):
    """
    CSV with ';' separator and latin-1 encoding:
      column 0 -> image path relative to the image folder
      column 1 -> class label (1-20), shifted to 0-19
    """

    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file, delimiter=";", encoding="latin-1")
        self.img_dir = img_dir
        self.transform = transform

        label_col = self.df.columns[1]
        self.df.dropna(subset=[label_col], inplace=True)
        self.df[label_col] = self.df[label_col].astype(int)

        self.image_paths = self.df.iloc[:, 0].apply(
            lambda x: os.path.join(img_dir, x.replace("\\", "/")))
        self.labels = self.df.iloc[:, 1]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.image_paths.iloc[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image, self.labels.iloc[idx] - 1
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return torch.zeros(3, 224, 224), -1   # removed by collate_fn


def collate_fn(batch):
    """Drop images that could not be loaded (label = -1)."""
    batch = [(x, y) for x, y in batch if y >= 0]
    if not batch:
        return torch.empty(0, 3, 224, 224), torch.empty(0, dtype=torch.int64)
    return torch.utils.data.default_collate(batch)


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=(5, 5))], p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
])


def eval_transform(resize=224):
    return transforms.Compose([
        transforms.Resize(resize),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
    ])


def split_70_15_15(n, seed):
    """
    70/15/15 split of the main experiment. As in the original notebook,
    random_split is called twice after setting the seed and the second call
    defines the split that was used.
    """
    set_seed(seed)
    lengths = [int(0.7 * n), int(0.15 * n)]
    lengths.append(n - sum(lengths))
    torch.utils.data.random_split(range(n), lengths)
    train, val, test = torch.utils.data.random_split(range(n), lengths)
    return train.indices, val.indices, test.indices

In [ ]:
# Full dataset with the evaluation transform; test set = same 70/15/15 split used for training
full_ds = FabricsDataset(CSV_FILE, IMG_DIR, transform=eval_transform())
_, _, test_idx = split_70_15_15(len(full_ds), SEED)
test_ds = Subset(full_ds, test_idx)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, collate_fn=collate_fn)

num_classes = full_ds.labels.nunique()
class_ids = np.arange(1, num_classes + 1)   # classes as numbered in the paper (1-20)
print(f"Test images: {len(test_ds)} | classes: {num_classes}")


def stem(path):
    """File name without folder and extension."""
    name = str(path).replace("\\", "/").split("/")[-1]
    return os.path.splitext(name)[0]


def find_image(name):
    """Index in full_ds of the image with this file name (with or without extension)."""
    matches = [i for i, p in enumerate(full_ds.df.iloc[:, 0]) if stem(p) == stem(name)]
    if not matches:
        raise ValueError(f"{name} not found in {CSV_FILE}")
    return matches[0]

In [ ]:
LVVIT_M = dict(patch_size=16, embed_dim=512, depth=20, num_heads=8, mlp_ratio=3.,
               p_emb="4_2", skip_lam=2., return_dense=True)
PRUNING_LOC = [5, 10, 15]
TOKEN_RATIO = [0.7, 0.49, 0.343]


def build_student(num_classes, pretrained_path=None):
    """DynamicViT LV-ViT-M student. ImageNet-pretrained weights are loaded before replacing the heads."""
    model = LVViTDiffPruning(**LVVIT_M, mix_token=True, pruning_loc=PRUNING_LOC,
                             token_ratio=TOKEN_RATIO, viz_mode=True)
    if pretrained_path:
        model.load_state_dict(torch.load(pretrained_path, map_location="cpu")["model"])
    model.head = nn.Linear(model.head.in_features, num_classes)
    model.aux_head = nn.Linear(model.aux_head.in_features, num_classes)
    return model


model = build_student(num_classes)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location="cpu"))
model.to(device).eval()

In [ ]:
# Predictions on the test set, shared by Fig. 3 and Fig. 4
probs_list, labels_list = [], []
with torch.no_grad():
    for images, labels in test_loader:
        if images.numel() == 0:
            continue
        logits, _ = model(images.to(device))
        probs_list.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
        labels_list.append(labels.numpy())

y_pred_probs = np.concatenate(probs_list)
y_true = np.concatenate(labels_list)
y_pred = y_pred_probs.argmax(1)
print(f"Test accuracy: {100 * (y_pred == y_true).mean():.2f}%")

## 3. Predictive entropy vs per-class accuracy (Fig. 3)

Normalised entropy of the softmax output (0 = certain, 1 = uniform), averaged
per class, against the per-class accuracy; marker size and colour give the
number of test images per class.

In [ ]:
from sklearn.metrics import confusion_matrix

eps = 1e-12
entropy = -(y_pred_probs * np.log(y_pred_probs + eps)).sum(1) / np.log(num_classes)

entropy_means = np.array([entropy[y_true == c].mean() for c in range(num_classes)])
counts_vals = np.array([(y_true == c).sum() for c in range(num_classes)])

cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
accuracy_cls = np.diag(cm) / np.clip(cm.sum(1), 1, None)

for c in range(num_classes):
    print(f"Class {c + 1}: images = {counts_vals[c]}, mean entropy = {entropy_means[c]:.4f}, "
          f"accuracy = {accuracy_cls[c]:.3f}")

In [ ]:
plt.figure(figsize=(6, 4))
sc = plt.scatter(entropy_means, accuracy_cls, c=counts_vals, s=40 + 2.5 * counts_vals,
                 cmap="inferno", edgecolor="black", linewidth=0.4, alpha=0.7)
cbar = plt.colorbar(sc)
cbar.set_label("Number of samples per class", size=16, labelpad=12)

# Label offsets (in points) chosen by hand to avoid overlaps; key = class index (0-based)
custom_offsets = {0: (-13, 0), 3: (4, 6), 4: (6, -2), 6: (6, 4), 7: (10, 0), 9: (2, 7),
                  10: (-6, 14), 11: (1, 10), 13: (10, 1), 14: (4, 6), 16: (10, -3),
                  17: (6, 4), 18: (-1, -13), 19: (8, -1)}
for i in range(num_classes):
    plt.annotate(str(i + 1), (entropy_means[i], accuracy_cls[i]),
                 xytext=custom_offsets.get(i, (7, 5)), textcoords="offset points", fontsize=7)

plt.xlabel("Mean predictive entropy", size=16, labelpad=12)
plt.ylabel("Per-class accuracy", size=16, labelpad=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig3_entropy_vs_accuracy.png"), dpi=400, bbox_inches="tight")
plt.show()

## 4. Calibration (Fig. 4)

Expected Calibration Error (ECE) with 10 equal-width confidence bins:
(a) reliability diagram, (b) ECE per class compared with the global ECE.

In [ ]:
def compute_ece(confidences, correct, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (confidences > lo) & (confidences <= hi)
        if in_bin.mean() > 0:
            ece += abs(confidences[in_bin].mean() - correct[in_bin].mean()) * in_bin.mean()
    return ece


def calibration_bins(y_true, y_probs, n_bins=10):
    """Accuracy, mean confidence and size of each confidence bin (reliability diagram)."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    confidences = y_probs.max(1)
    correct = y_probs.argmax(1) == y_true
    accs, confs, sizes = [], [], []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (confidences >= lo) & (confidences <= hi) if i == 0 else (confidences > lo) & (confidences <= hi)
        sizes.append(in_bin.sum())
        accs.append(correct[in_bin].mean() if in_bin.any() else 0.0)
        confs.append(confidences[in_bin].mean() if in_bin.any() else 0.0)
    return np.array(accs), np.array(confs), bin_edges, np.array(sizes)


n_bins = 10
confidences = y_pred_probs.max(1)
correct = y_pred == y_true
ece = compute_ece(confidences, correct, n_bins)

ece_per_class = np.array([
    compute_ece(confidences[y_true == c], correct[y_true == c], n_bins) if (y_true == c).any() else np.nan
    for c in range(num_classes)
])
bin_accs, bin_confs, bin_boundaries, bin_sizes = calibration_bins(y_true, y_pred_probs, n_bins)
print(f"Global ECE: {ece:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.5), facecolor="white",
                         gridspec_kw={"width_ratios": [1, 1.6]})

# (a) Reliability diagram
ax = axes[0]
bin_width = 1.0 / n_bins
x_coords = bin_boundaries[:-1] + bin_width / 2
ax.bar(x_coords, np.where(bin_sizes > 0, bin_accs, np.nan), width=bin_width * 0.9, align="center",
       color="#3A6EA5", alpha=0.70, edgecolor="black", linewidth=0.6, zorder=2, label="Outputs")
gap_label_added = False
for i in range(n_bins):
    if bin_sizes[i] > 0:
        ax.bar(x_coords[i], bin_confs[i] - bin_accs[i], width=bin_width * 0.9, bottom=bin_accs[i],
               align="center", color="darkred", alpha=0.35, edgecolor="black", hatch="//",
               linewidth=0.5, label="" if gap_label_added else "Gap", zorder=3)
        gap_label_added = True
ax.plot([0, 1], [0, 1], linestyle="--", color="#444444", linewidth=2, label="Calibration", zorder=10)
ax.set_xlabel("Confidence", fontsize=20, labelpad=10)
ax.set_ylabel("Accuracy", fontsize=20, labelpad=10)
ax.set_xticks(np.arange(0, 1.1, 0.2))
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.tick_params(axis="both", labelsize=14)
ax.legend(fontsize=14, frameon=False)
ax.text(-0.18, 1.00, "(a)", transform=ax.transAxes, fontsize=16, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# (b) Class-wise ECE
ax = axes[1]
ax.bar(class_ids, ece_per_class, color="#3A6EA5", alpha=0.7, edgecolor="black", linewidth=0.6, zorder=2)
ax.axhline(y=ece, color="darkred", linestyle="--", linewidth=2, label=f"Global ECE: {ece:.4f}", zorder=3)
ax.set_xlabel("Classes", fontsize=20, labelpad=10)
ax.set_ylabel("Expected Calibration Error (ECE)", fontsize=20, labelpad=10)
ax.set_xticks(class_ids)
ax.tick_params(axis="both", labelsize=14)
ax.legend(fontsize=14, frameon=False, loc="upper left")
ax.text(-0.065, 1.0, "(b)", transform=ax.transAxes, fontsize=16, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig4_calibration.png"), dpi=400, bbox_inches="tight")
plt.show()

## 5. Token pruning: helper functions

The image (224×224) is split into 14×14 = 196 patches of 16×16 pixels (tokens).
At each pruning stage (layers 5, 10, 15) the model keeps a subset of tokens;
`collect_keep_indices` maps the kept tokens back to their original positions.

In [ ]:
GRID = 14            # 14 x 14 tokens
PATCH = 16           # 16 x 16 pixels per token
NUM_TOKENS = GRID * GRID


def denormalize(image):
    mean = torch.tensor(IMAGENET_DEFAULT_MEAN, device=image.device).view(3, 1, 1)
    std = torch.tensor(IMAGENET_DEFAULT_STD, device=image.device).view(3, 1, 1)
    return (image * std + mean).clamp(0, 1)


def collect_keep_indices(decisions, b_idx=0):
    """Absolute positions (0..195) of the tokens kept after each pruning stage."""
    alive = np.arange(NUM_TOKENS)
    keep_indices = []
    for stage in range(len(decisions)):
        rel_idx = decisions[stage][0][b_idx].detach().cpu().numpy()
        alive = alive[rel_idx]
        keep_indices.append(alive.copy())
    return keep_indices


def to_tokens(img_np):
    """(224, 224, 3) image -> (196, 16, 16, 3) patches."""
    return img_np.reshape(GRID, PATCH, GRID, PATCH, 3).swapaxes(1, 2).reshape(NUM_TOKENS, PATCH, PATCH, 3)


def recover_image(tokens):
    """(196, 16, 16, 3) patches -> (224, 224, 3) image."""
    return tokens.reshape(GRID, GRID, PATCH, PATCH, 3).swapaxes(1, 2).reshape(GRID * PATCH, GRID * PATCH, 3)


def gen_masked_tokens(tokens, kept_abs_idx, alpha=0.6):
    """Whiten the pruned tokens (alpha: 0 = original, 1 = white)."""
    tokens = tokens.copy().astype(np.float32)
    drop_idx = np.setdiff1d(np.arange(NUM_TOKENS), kept_abs_idx)
    tokens[drop_idx] = (1 - alpha) * tokens[drop_idx] + alpha * 255.0
    return tokens.astype(np.uint8)


def forward_single(idx):
    """Image idx of full_ds -> (image tensor on device, logits, pruning decisions)."""
    img, label = full_ds[idx]
    img = img.unsqueeze(0).to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
        logits, decisions = model(img)
    return img, logits, decisions, label

## 6. Sparsification process for the 20 fabrics (Fig. 5)

One representative XPL image per class (selected from the whole dataset),
shown as original input and after the three pruning stages.

In [ ]:
FIG5_FILES = [
    "J-EB-12-4X-XPL.tif",      # Class 1
    "DA-94-10X-XPL.tif",       # Class 2
    "718-25-2,5X-XPL.png",     # Class 3
    "camp15_XPL_10X_1.png",    # Class 4
    "K9_XPL_10X.png",          # Class 5
    "J-19-4X-XPL.tif",         # Class 6
    "KB_54_220-XPL-10X_1.png", # Class 7
    "X.706-XPL.tif",           # Class 8
    "E822-29-10X-XPL_1.png",   # Class 9
    "E989-37-2,5X-XPL_1.png",  # Class 10
    "DA-154-10X-XPL_1.tif",    # Class 11
    "TB-4-4X-XPL_1.tif",       # Class 12
    "DA-144-10X-XPL_1.tif",    # Class 13
    "DA-BA-85-10X-XPL_1.tif",  # Class 14
    "JA-11-10X-XPL_1.tif",     # Class 15
    "J-EB-49-4X-XPL.tif",      # Class 16
    "P.2517-XPL.2.tif",        # Class 17
    "X.689-XPL.3.tif",         # Class 18
    "T9-2,5X-XPL.png",         # Class 19
    "DA-87-4X-XPL.tif",        # Class 20
]

stages = len(PRUNING_LOC)
rows_per_col = num_classes // 2          # two blocks of 10 classes side by side
fig, axes = plt.subplots(rows_per_col, (stages + 1) * 2,
                         figsize=(4 * (stages + 1) * 2, 4 * rows_per_col))

for cls_idx, fname in enumerate(FIG5_FILES):
    row = cls_idx % rows_per_col
    col_offset = (cls_idx // rows_per_col) * (stages + 1)

    img, _, decisions, _ = forward_single(find_image(fname))
    keep_indices = collect_keep_indices(decisions)
    img_np = (denormalize(img[0]).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    tokens = to_tokens(img_np)

    ax = axes[row, col_offset]
    ax.imshow(img_np)
    ax.set_title(f"Class {cls_idx + 1}", fontsize=34, pad=3)
    for s in range(stages):
        ax = axes[row, col_offset + s + 1]
        ax.imshow(recover_image(gen_masked_tokens(tokens, keep_indices[s])))
        if row == 0:
            ax.set_title(f"Stage {s + 1}\n{len(keep_indices[s])} patches", fontsize=34)

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(pad=1.0)
plt.savefig(os.path.join(FIG_DIR, "fig5_pruning_per_class.png"), dpi=300, bbox_inches="tight")
plt.show()

## 7. Token keep probability (Fig. 6)

Fraction of test images in which each token is kept at each pruning stage,
averaged over 2×2 blocks of tokens (7×7 grid).

In [ ]:
import matplotlib.gridspec as gridspec

num_stages = len(PRUNING_LOC)
counts_global = np.zeros((num_stages, NUM_TOKENS), dtype=np.int64)
num_images_total = 0

with torch.no_grad():
    for imgs, _ in test_loader:
        if imgs.numel() == 0:
            continue
        _, decisions = model(imgs.to(device))
        num_images_total += imgs.size(0)
        for b in range(imgs.size(0)):
            alive = np.arange(NUM_TOKENS)
            for stage in range(num_stages):
                alive = alive[decisions[stage][0][b].detach().cpu().numpy()]
                counts_global[stage, alive] += 1

fig = plt.figure(figsize=(18, 6))
gs = gridspec.GridSpec(1, num_stages + 1, width_ratios=[1] * num_stages + [0.04], wspace=0.05)
cmap = "inferno"

for stage in range(num_stages):
    ax = fig.add_subplot(gs[0, stage])
    prob_grid = (counts_global[stage] / num_images_total).reshape(GRID, GRID)
    prob_grid = prob_grid.reshape(7, 2, 7, 2).mean(axis=(1, 3))     # 7 x 7 pooling
    sns.heatmap(prob_grid, square=True, cbar=False, annot=True, fmt=".2f", annot_kws={"size": 10},
                linewidths=0.1, linecolor="white", vmin=0, vmax=1, cmap=cmap, ax=ax)
    avg_kept = counts_global[stage].sum() / num_images_total
    ax.set_title(f"Keep Prob - Stage {PRUNING_LOC[stage]}\nAvg kept: {avg_kept:.1f}/{NUM_TOKENS}", size=16)
    ax.axis("off")

cax = fig.add_subplot(gs[0, -1])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
cbar = plt.colorbar(sm, cax=cax)
cbar.set_label("Keep Probability", rotation=270, size=20, labelpad=30)
cbar.ax.tick_params(labelsize=14)

plt.savefig(os.path.join(FIG_DIR, "fig6_token_keep_probability.png"), dpi=300, bbox_inches="tight")
plt.show()

## 8. Token exit depth and token influence (Fig. 7)

For five test images:
- **Exit depth**: layer at which each token is pruned (5, 10, 15; tokens kept
  until the end are assigned 15).
- **Token influence (Δprob)**: drop in the probability of the predicted class
  when a single 16×16 patch is set to zero (one forward pass per patch,
  196 per image), smoothed and normalised per image.

In [ ]:
from skimage.transform import resize
from scipy.ndimage import gaussian_filter

FIG7_FILES = {   # class (1-20): image from the test set
    1:  "KB_248_2-XPL-2,5X",
    8:  "X.636-XPL.2",
    11: "DA-155-10X-XPL_1",
    13: "DA-162-4X-XPL",
    15: "HT-8-10X-XPL",
}


def compute_exit_depth(decisions, b_idx=0):
    alive = np.arange(NUM_TOKENS)
    exit_depth = np.zeros(NUM_TOKENS, dtype=np.int32)
    for stage, keep in enumerate(decisions):
        alive_next = alive[keep[0][b_idx].detach().cpu().numpy()]
        exit_depth[np.setdiff1d(alive, alive_next, assume_unique=True)] = PRUNING_LOC[stage]
        alive = alive_next
    exit_depth[alive] = PRUNING_LOC[-1]
    return exit_depth


def prune_single_patch(img, patch_idx):
    img = img.clone()
    r, c = divmod(patch_idx, GRID)
    img[:, :, r * PATCH:(r + 1) * PATCH, c * PATCH:(c + 1) * PATCH] = 0
    return img


test_positions = set(test_idx)
fig, axes = plt.subplots(len(FIG7_FILES), 3, figsize=(18, 6 * len(FIG7_FILES)))
axes = np.atleast_2d(axes)

for row, (cls, fname) in enumerate(FIG7_FILES.items()):
    idx = find_image(fname)
    if idx not in test_positions:
        print(f"Warning: {fname} is not in the test set")
    img, logits, decisions, label = forward_single(idx)
    assert label + 1 == cls, f"{fname} belongs to class {label + 1}, not {cls}"
    img_np = denormalize(img[0]).permute(1, 2, 0).cpu().numpy()

    # Exit depth
    exit_grid = compute_exit_depth(decisions).reshape(GRID, GRID)
    exit_resized = resize(exit_grid.astype(np.float32), img_np.shape[:2], order=0,
                          mode="reflect", anti_aliasing=False)

    # Token influence (delta prob)
    probs = torch.softmax(logits[0].float(), dim=-1)
    target_class = probs.argmax().item()
    orig_target = probs[target_class].item()
    delta_token = np.zeros(NUM_TOKENS, dtype=np.float32)
    for i in range(NUM_TOKENS):
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=img.is_cuda):
            out = model(prune_single_patch(img, i))
            new_logits = out[0] if isinstance(out, tuple) else out
        new_prob = torch.softmax(new_logits[0].float(), dim=-1)[target_class].item()
        delta_token[i] = max(0, orig_target - new_prob)

    delta_map = resize(delta_token.reshape(GRID, GRID), img_np.shape[:2], order=5,
                       mode="reflect", anti_aliasing=True)
    delta_map = gaussian_filter(delta_map, sigma=8)
    delta_norm = delta_map / (delta_map.max() + 1e-8) if delta_map.max() > 0 else delta_map

    # Plots
    ax_row = axes[row]
    ax_row[0].imshow(img_np)
    ax_row[0].set_title(f"Original (Class {cls})", size=28, pad=10)

    ax_row[1].imshow(img_np)
    im_exit = ax_row[1].imshow(exit_resized, cmap="viridis", alpha=0.4)
    ax_row[1].set_title("Overlay Exit Depth", size=28, pad=10)
    fig.colorbar(im_exit, ax=ax_row[1], fraction=0.046, pad=0.04).ax.tick_params(labelsize=18)

    colored = plt.colormaps["inferno"](delta_norm)
    colored[..., -1] = np.clip(delta_norm * 1.5, 0.1, 0.8)
    ax_row[2].imshow(img_np)
    ax_row[2].imshow(colored)
    ax_row[2].set_title("Overlay Token Influence", size=28, pad=10)
    fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0, 1), cmap="inferno"),
                 ax=ax_row[2], fraction=0.046, pad=0.04).ax.tick_params(labelsize=18)

    for ax in ax_row:
        ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig7_exit_depth_token_influence.png"), dpi=300, bbox_inches="tight")
plt.show()